In [ ]:
import pandas as pd

In [ ]:
from google.colab import files
uploaded_files = files.upload()

In [ ]:
df = pd.read_csv("spam.csv", encoding="latin-1")
df

In [ ]:
if 'v1' in df.columns and 'v2' in df.columns:
    df = df[['v1', 'v2']]
    # rename columns
    df.columns = ['label', 'message']
elif 'label' in df.columns and 'message' in df.columns:
    # If columns are already renamed, do nothing or re-display head
    pass
else:
    # Handle unexpected column names if neither original nor renamed exist
    print("DataFrame does not contain expected columns ('v1', 'v2' or 'label', 'message').")

df.head()

In [ ]:
df['label'] = df['label'].map({
    'ham': 0,
    'spam': 1
})

df.head()

In [ ]:
df['label'].value_counts()

In [ ]:
df['message'] = df['message'].str.lower()
df.head()

In [ ]:
import string

In [ ]:
df['message'] = df['message'].astype(str).apply(lambda x : x.translate(str.maketrans('', '', string.punctuation)))
df.head()

In [ ]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

In [ ]:
df['message'] = df['message'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stopwords.words('english'))]))
df.head()

In [ ]:
from nltk.stem import PorterStemmer

ps = PorterStemmer()

# Apply stemming to each message in the 'message' column
df['message'] = df['message'].apply(lambda text: ' '.join([ps.stem(word) for word in text.split()]))
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X = df['message']
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
vectorizer = TfidfVectorizer()

X_train = vectorizer.fit_transform(X_train)

X_test = vectorizer.transform(X_test)

In [ ]:
# Print the shape of the TF-IDF matrix for the training data
print(f"Shape of X_train TF-IDF matrix: {X_train.shape}")

# The shape tells us: (number of documents, number of unique words/features)

In [ ]:
# Get feature names (words) from the vectorizer
feature_names = vectorizer.get_feature_names_out()

# Display a few feature names
print("\nFirst 20 feature names (words):")
print(feature_names[:20])

To see the actual numerical values, let's look at the TF-IDF scores for the first message and the corresponding words. We'll convert the first row of the sparse matrix to a dense array and then map it back to the words.

In [ ]:
# Get the TF-IDF scores for the first message in the training set
# Use X_train[0] because the vectorizer and feature_names were derived from the training data.
first_message_tfidf = X_train[0].toarray()

# Create a pandas Series for easier viewing of word-score pairs
# feature_names should correspond to the columns of X_train
word_scores = pd.Series(first_message_tfidf[0], index=feature_names)

# Display the words with their non-zero TF-IDF scores for the first message
print("\nTF-IDF scores for the first message (non-zero values only):")
display(word_scores[word_scores > 0].sort_values(ascending=False))

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)
y_train_smote.value_counts()

In [ ]:
from sklearn.naive_bayes import MultinomialNB
model = MultinomialNB()

model.fit(X_train_smote, y_train_smote)

In [ ]:
y_pred = model.predict(X_test)
y_pred

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

In [ ]:
y_pred[:10]

In [ ]:
y_test[:10].values

In [ ]:
msg = ["Your Amazon order has been delivered"]
msg_vector = vectorizer.transform(msg)

prediction = model.predict(msg_vector)

print(prediction)

In [ ]:
if prediction[0] == 1:
    print("Spam")
else:
    print("Not Spam")